# 06 — Cloud & Foundry Evaluation

**What you'll learn:**
- Logging local evaluation runs to your Azure AI Foundry project
- Viewing results in the Foundry portal
- When to use cloud (remote) evaluation via `azure-ai-projects`

---

Local runs are great for the inner loop, but teams need **shared, auditable** results.
Passing `azure_ai_project` to `evaluate()` uploads the run to your Foundry project so
anyone can inspect every row, metric, and reason in the portal.

In [ ]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

## Log a local run to Foundry

Add `azure_ai_project=` (your project endpoint) to any `evaluate()` call. The run
executes locally, then results are pushed to the project's **Evaluation** tab.

In [ ]:
from azure.ai.evaluation import evaluate, RelevanceEvaluator, GroundednessEvaluator

assert azure_ai_project, "Set AZURE_AI_PROJECT_ENDPOINT in your .env"

result = evaluate(
    data=f"{DATA_DIR}/grounded-context.jsonl",
    evaluators={
        "relevance": RelevanceEvaluator(model_config),
        "groundedness": GroundednessEvaluator(model_config),
    },
    azure_ai_project=azure_ai_project,
    evaluation_name="banking-qa-local-logged",
)

# The response includes a portal URL for the logged run.
result.get("studio_url")

## View results in the portal

Open your project in [Azure AI Foundry](https://ai.azure.com) → **Evaluation**. You'll
see the run `banking-qa-local-logged` with:

- aggregate metric cards
- a per-row table (query, response, each score + reasoning)
- run metadata for comparison across prompt/model versions

Compare two runs side-by-side to prove a prompt change actually improved quality.

## Remote (cloud) evaluation with `azure-ai-projects`

The run above executed on *your* machine and only *logged* to Foundry. For large
datasets or scheduled runs you can have **Foundry execute the evaluation server-side**
using `azure-ai-projects`. The sketch below shows the shape — see the SDK docs for the
current API surface.

In [ ]:
# Illustrative — remote evaluation runs the job in Foundry, not locally.
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=azure_ai_project,
    credential=DefaultAzureCredential(),
)

# 1. Upload the dataset to the project.
# 2. Create an evaluation referencing the dataset + evaluator IDs.
# 3. Poll for completion, then read metrics — all server-side.
#
# See: https://learn.microsoft.com/azure/ai-foundry/how-to/develop/cloud-evaluation
print("Project client ready:", project is not None)

## Local vs cloud — which when?

| | Local (`evaluate` + `azure_ai_project`) | Cloud (`azure-ai-projects`) |
|---|---|---|
| Runs on | your machine / CI runner | Foundry service |
| Best for | dev loop, CI gates, small data | large datasets, scheduled runs |
| Results in Foundry | yes (logged) | yes (native) |

## Recap

- `azure_ai_project=` logs local runs to Foundry for shared, auditable results.
- The portal's Evaluation tab shows per-row scores + reasoning and run comparisons.
- Use `azure-ai-projects` for server-side runs on large or scheduled workloads.

**Next:** [07 — CI automated evaluation](./07-ci-automated-evaluation.ipynb) — gate every
PR with the GenAI Evaluation GitHub Action.